In [45]:
#============================================================
# CUSTOMER LOOKUP ETL PIPELINE
# ============================================================
# Author : Senior Data Engineer / Python Data Analyst
# Purpose:
#   - Extract customer data from MySQL
#   - Clean and transform the dataset
#   - Save cleaned data to CSV
# ============================================================

# ----------------------------
# Import Required Libraries
# ----------------------------
import os
import re
from datetime import datetime

import mysql.connector
import pandas as pd


In [47]:
# ============================================================
# MYSQL DATABASE CONFIGURATION
# ============================================================
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '1234',
    'database': 'adventureworks'
}


In [48]:
# ============================================================
# EMAIL VALIDATION FUNCTION
# ============================================================
def validate_email(email):
    """
    Validate email using regex pattern.
    """

    if pd.isna(email):
        return None

    email = str(email).strip()

    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    if re.match(pattern, email):
        return email

    return None


In [49]:


# ============================================================
# INCOME CATEGORY FUNCTION
# ============================================================
def income_category(income):
    """
    Categorize annual income.
    """

    if pd.isna(income):
        return 'Unknown'

    if income < 50000:
        return 'Low'

    elif income < 100000:
        return 'Medium'

    else:
        return 'High'



In [50]:
# ============================================================
# MAIN ETL FUNCTION
# ============================================================
def main():

    connection = None
    cursor = None

    try:

        # ====================================================
        # STEP 1: CONNECT TO MYSQL DATABASE
        # ====================================================
        print("=" * 60)
        print("CONNECTING TO MYSQL DATABASE")
        print("=" * 60)

        connection = mysql.connector.connect(**DB_CONFIG)

        if connection.is_connected():
            print("Database connection successful.\n")

        # ====================================================
        # STEP 2: EXTRACT DATA FROM MYSQL TABLE
        # ====================================================
        print("=" * 60)
        print("EXTRACTING DATA FROM customer_lookup TABLE")
        print("=" * 60)

        query = "SELECT * FROM customer_lookup"

        # Create cursor
        cursor = connection.cursor(dictionary=True)

        # Execute query
        cursor.execute(query)

        # Fetch all records
        rows = cursor.fetchall()

        # Convert records to DataFrame
        df = pd.DataFrame(rows)

        print("Data extraction completed successfully.\n")

        # ====================================================
        # ROW COUNT BEFORE CLEANING
        # ====================================================
        rows_before_cleaning = len(df)

        print(f"Rows Before Cleaning : {rows_before_cleaning}")

        # ====================================================
        # STEP 3: DATA CLEANING
        # ====================================================

        # ----------------------------------------------------
        # REMOVE DUPLICATE ROWS
        # ----------------------------------------------------
        duplicate_count = df.duplicated().sum()

        df.drop_duplicates(inplace=True)

        print(f"Duplicate Rows Removed : {duplicate_count}")

        # ----------------------------------------------------
        # REMOVE EXTRA SPACES FROM STRING COLUMNS
        # ----------------------------------------------------
        string_columns = df.select_dtypes(include='object').columns

        for col in string_columns:
            df[col] = df[col].astype(str).str.strip()

        # ----------------------------------------------------
        # REPLACE EMPTY STRINGS WITH NULL VALUES
        # ----------------------------------------------------
        df.replace(r'^\s*$', pd.NA, regex=True, inplace=True)

         # ----------------------------------------------------
        # STANDARDIZE TEXT COLUMNS TO PROPER CASE
        # ----------------------------------------------------
        proper_case_columns = [
            'Prefix',
            'FirstName',
            'LastName',
            'EducationLevel',
            'Occupation'
        ]

        for col in proper_case_columns:

            if col in df.columns:

                df[col] = (
                    df[col]
                    .astype(str)
                    .str.title()
                )

        # ----------------------------------------------------
        # VALIDATE EMAIL ADDRESSES
        # ----------------------------------------------------
        df['EmailAddress'] = df['EmailAddress'].apply(
            validate_email
        )

        # ----------------------------------------------------
        # CONVERT BIRTHDATE TO DATETIME FORMAT
        # ----------------------------------------------------
        df['BirthDate'] = pd.to_datetime(
            df['BirthDate'],
            errors='coerce'
        )

        # ----------------------------------------------------
        # REMOVE IMPOSSIBLE BIRTH DATES
        # CONDITIONS:
        #   - Not future date
        #   - Age <= 120 years
        # ----------------------------------------------------
        today = pd.Timestamp.today()

        min_birth_date = today - pd.DateOffset(years=120)

        df = df[
            (df['BirthDate'].notna()) &
            (df['BirthDate'] <= today) &
            (df['BirthDate'] >= min_birth_date)
        ]

        # ----------------------------------------------------
        # STANDARDIZE GENDER VALUES
        # ----------------------------------------------------
        gender_mapping = {
            'M': 'Male',
            'MALE': 'Male',
            'F': 'Female',
            'FEMALE': 'Female',
            'O': 'Other',
            'OTHER': 'Other'
        }

        df['Gender'] = (
            df['Gender']
            .astype(str)
            .str.upper()
            .map(gender_mapping)
        )

        df['Gender'].fillna('Other', inplace=True)

        # ----------------------------------------------------
        # STANDARDIZE MARITAL STATUS VALUES
        # ----------------------------------------------------
        marital_mapping = {
            'M': 'Married',
            'MARRIED': 'Married',
            'S': 'Single',
            'SINGLE': 'Single',
            'D': 'Divorced',
            'DIVORCED': 'Divorced',
            'W': 'Widowed',
            'WIDOWED': 'Widowed'
        }

        df['MaritalStatus'] = (
            df['MaritalStatus']
            .astype(str)
            .str.upper()
            .map(marital_mapping)
        )

        df['MaritalStatus'].fillna('Unknown', inplace=True)

        # ----------------------------------------------------
        # CONVERT AnnualIncome TO NUMERIC
        # ----------------------------------------------------
        df['AnnualIncome'] = pd.to_numeric(
            df['AnnualIncome'],
            errors='coerce'
        )

        # ----------------------------------------------------
        # HANDLE NEGATIVE OR INVALID INCOME VALUES
        # ----------------------------------------------------
        df.loc[df['AnnualIncome'] < 0, 'AnnualIncome'] = pd.NA

        median_income = df['AnnualIncome'].median()

        df['AnnualIncome'].fillna(
            median_income,
            inplace=True
        )

        # ----------------------------------------------------
        # CONVERT TotalChildren TO INTEGER
        # ----------------------------------------------------
        df['TotalChildren'] = pd.to_numeric(
            df['TotalChildren'],
            errors='coerce'
        )

        # ----------------------------------------------------
        # HANDLE INVALID CHILDREN COUNTS
        # ----------------------------------------------------
        df.loc[df['TotalChildren'] < 0, 'TotalChildren'] = pd.NA

        df.loc[df['TotalChildren'] > 20, 'TotalChildren'] = pd.NA

        df['TotalChildren'].fillna(0, inplace=True)

        df['TotalChildren'] = df['TotalChildren'].astype(int)

        # ----------------------------------------------------
        # NORMALIZE HomeOwner VALUES
        # ----------------------------------------------------
        homeowner_mapping = {
            'Y': 'Yes',
            'YES': 'Yes',
            'N': 'No',
            'NO': 'No',
            '1': 'Yes',
            '0': 'No',
            'TRUE': 'Yes',
            'FALSE': 'No'
        }

        df['HomeOwner'] = (
            df['HomeOwner']
            .astype(str)
            .str.upper()
            .map(homeowner_mapping)
        )

        df['HomeOwner'].fillna('No', inplace=True)

        # ====================================================
        # STEP 4: DATA TRANSFORMATION
        # ====================================================

        # ----------------------------------------------------
        # CREATE FullName COLUMN
        # ----------------------------------------------------
        df['FullName'] = (
            df['FirstName'].fillna('') + ' ' +
            df['LastName'].fillna('')
        ).str.strip()

        # ----------------------------------------------------
        # CREATE Age COLUMN
        # ----------------------------------------------------
        df['Age'] = (
            (today - df['BirthDate']).dt.days // 365
        )

        # ----------------------------------------------------
        # CREATE IncomeCategory COLUMN
        # ----------------------------------------------------
        df['IncomeCategory'] = df['AnnualIncome'].apply(
            income_category
        )

        # ----------------------------------------------------
        # RENAME COLUMNS TO snake_case
        # ----------------------------------------------------
        df.columns = [
            col.strip()
               .replace(" ", "_")
               .replace("-", "_")
               .lower()
            for col in df.columns
        ]

        # ====================================================
        # STEP 5: DISPLAY SUMMARY STATISTICS
        # ====================================================
        print("\n" + "=" * 60)
        print("SUMMARY STATISTICS")
        print("=" * 60)

        print("\nDataFrame Information:")
        print(df.info())

        print("\nNull Value Counts:")
        print(df.isnull().sum())

        print("\nDescriptive Statistics:")
        print(df.describe(include='all'))

        # ====================================================
        # FINAL ROW COUNT
        # ====================================================
        rows_after_cleaning = len(df)

        print(f"\nRows After Cleaning : {rows_after_cleaning}")

        # ====================================================
        # STEP 6: CREATE OUTPUT DIRECTORY
        # ====================================================
        output_directory = os.path.dirname(
            OUTPUT_FILE_PATH
        )

        os.makedirs(output_directory, exist_ok=True)

        # ====================================================
        # STEP 7: SAVE CLEANED DATA TO CSV
        # ====================================================
        df.to_csv(
            OUTPUT_FILE_PATH,
            index=False
        )

        print("\nCleaned CSV file saved successfully.")

        print(f"\nCSV File Location:\n{OUTPUT_FILE_PATH}")

    # ========================================================
    # DATABASE ERROR HANDLING
    # ========================================================
    except mysql.connector.Error as db_error:

        print("\nMYSQL DATABASE ERROR:")
        print(db_error)

    # ========================================================
    # FILE PATH ERROR HANDLING
    # ========================================================
    except FileNotFoundError as file_error:

        print("\nFILE PATH ERROR:")
        print(file_error)

    # ========================================================
    # PERMISSION ERROR HANDLING
    # ========================================================
    except PermissionError as permission_error:

        print("\nPERMISSION ERROR:")
        print(permission_error)

    # ========================================================
    # GENERAL ERROR HANDLING
    # ========================================================
    except Exception as e:

        print("\nUNEXPECTED ERROR OCCURRED:")
        print(e)

    # ========================================================
    # CLOSE DATABASE CONNECTION
    # ========================================================
    finally:

        # Close cursor
        if cursor is not None:
            cursor.close()

        # Close database connection
        if connection is not None and connection.is_connected():

            connection.close()

            print("\nMySQL connection closed.")




In [51]:
# ============================================================
# OUTPUT CSV FILE PATH
# ============================================================
OUTPUT_FILE_PATH = (
    r'C:\Users\Radhika Kashid\OneDrive\Documents'
    r'\DSA_End to end\PythonCleanedData'
    r'\customer_lookup_cleaned.csv'
)

In [52]:
# ============================================================
# EXECUTE MAIN FUNCTION
# ============================================================
if __name__ == "__main__":
    main()

CONNECTING TO MYSQL DATABASE
Database connection successful.

EXTRACTING DATA FROM customer_lookup TABLE
Data extraction completed successfully.

Rows Before Cleaning : 18148
Duplicate Rows Removed : 0


C:\Users\Radhika Kashid\AppData\Local\Temp\ipykernel_30668\4217492989.py:150: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Gender'].fillna('Other', inplace=True)
C:\Users\Radhika Kashid\AppData\Local\Temp\ipykernel_30668\4217492989.py:173: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 


SUMMARY STATISTICS

DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18148 entries, 0 to 18147
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerkey     18148 non-null  int64         
 1   prefix          18148 non-null  object        
 2   firstname       18148 non-null  object        
 3   lastname        18148 non-null  object        
 4   birthdate       18148 non-null  datetime64[ns]
 5   maritalstatus   18148 non-null  object        
 6   gender          18148 non-null  object        
 7   emailaddress    18109 non-null  object        
 8   annualincome    18148 non-null  float64       
 9   totalchildren   18148 non-null  int32         
 10  educationlevel  18148 non-null  object        
 11  occupation      18148 non-null  object        
 12  homeowner       18148 non-null  object        
 13  fullname        18148 non-null  object        
 14  age       